<a href="https://colab.research.google.com/github/JoaciQueiroz/Tutor_Idiomas_Interativo/blob/main/Dio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
language = 'pt'

# Etapa 1: Captura de Áudio (Frontend)

In [ ]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
import time

# Código JavaScript para capturar áudio
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
    print("🎤 Preparando para gravar...")
    for i in range(3, 0, -1):
        print(f"⏳ {i}...")
        time.sleep(1)
    print(f"▶️ Gravando áudio por {sec} segundos... fale agora!")

    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = b64decode(js_result.split(',')[1])
    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)

    print("✅ Gravação concluída! Processando o áudio...")
    return f'/content/{file_name}'

# Grava o áudio
record_file = record()
display(Audio(record_file, autoplay=False))

# Etapa 2: Reconhecimento de fala com Vosk (gratuito)

In [ ]:
!pip install vosk soundfile pydub googletrans==4.0.0-rc1 gtts
!apt-get update && apt-get install -y ffmpeg

from pydub import AudioSegment

input_audio_path = "request_audio.wav"
output_audio_path = "converted_audio.wav"

try:
    audio = AudioSegment.from_file(input_audio_path)
    audio = audio.set_channels(1).set_frame_rate(16000).set_sample_width(2)
    audio.export(output_audio_path, format="wav")
    print("✅ Áudio convertido para WAV compatível:", output_audio_path)
except Exception as e:
    print("❌ Erro ao converter o áudio:", e)

# Etapa 3: Interpretação com ChatGPT

In [ ]:
import wave, json
from vosk import Model, KaldiRecognizer

# Baixar e descompactar modelo (se ainda não estiver no ambiente)
!wget -nc https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip
!unzip -o vosk-model-small-en-us-0.15.zip

# Carregar modelo
model = Model("vosk-model-small-en-us-0.15")
wf = wave.open("converted_audio.wav", "rb")
rec = KaldiRecognizer(model, wf.getframerate())

results = []
while True:
    data = wf.readframes(4000)
    if len(data) == 0:
        break
    if rec.AcceptWaveform(data):
        results.append(json.loads(rec.Result()))

final_result = json.loads(rec.FinalResult())
results.append(final_result)

# Concatenar texto
user_text = " ".join([res.get("text", "") for res in results]).strip()

# Garantir que não seja vazio
if not user_text:
    #user_text = "Nenhum áudio reconhecido."
    user_text = "Passaro pequeno azul."

# Mostrar transcrição final de forma destacada
print("\n==============================")
print("📝 TRANSCRIÇÃO FINAL DO ÁUDIO")
print("==============================")
print(user_text)
print("==============================\n")

# Etapa 4: Conversão em Voz (gTTS)

In [ ]:
from googletrans import Translator
translator = Translator()

if user_text.strip():
    translation = translator.translate(user_text, src="pt", dest="en")
    answer = translation.text
else:
    answer = "Blue House."

print("\n💡 Tradução para inglês:")
print(answer)

# Etapa 5: Reprodução da Resposta (Frontend)

In [ ]:
!pip install gtts
!pip install --upgrade click

from gtts import gTTS

tts = gTTS(answer, lang="en")
output_path = "response.mp3"
tts.save(output_path)
print("🔊 Resposta convertida em áudio com gTTS.")

# Etapa 6: Reprodução

In [ ]:
display(Audio(output_path, autoplay=True))